# 06 — Validacao Layer-by-Layer

Validacao automatizada de todas as camadas do pipeline OCI:

| # | Camada | Checks |
|---|--------|--------|
| 1 | **Bronze** | 9 tabelas existem, row count > 0 |
| 2 | **Silver rawdata** | 19 tabelas existem, PK uniqueness |
| 3 | **Gold books** | 3 books existem, prefix check, row count |
| 4 | **Gold consolidated** | col count, SAFRA, leakage, targets, PK |
| 5 | **Gold final** | col count == features + keys, row count == consolidated |
| 6 | **Cross-layer** | Bronze >= Silver, Gold rows == Silver base rows |

**Resultado**: Relatorio PASS/FAIL por check.

**Hackathon PoD Academy** — Claro + Oracle

In [ ]:
import sys
import os
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count as spark_count, isnan

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from config import *
from utils import *

In [ ]:
# SparkSession com Delta Lake
builder = SparkSession.builder.appName("06-layer-validation")
for k, v in SPARK_DELTA_CONFIG.items():
    builder = builder.config(k, v)
spark = builder.getOrCreate()
print(f"Spark {spark.version} iniciado com Delta Lake")

In [ ]:
# Funcoes auxiliares
results = []


def read_table(spark_session, uri, fmt="delta"):
    """Le tabela no formato especificado."""
    if fmt == "delta":
        return spark_session.read.format("delta").load(uri)
    return spark_session.read.parquet(uri)


def add_result(check_name, layer, status, detail=""):
    """Adiciona resultado de validacao."""
    results.append({"check": check_name, "layer": layer, "status": status, "detail": detail})
    icon = "PASS" if status == "PASS" else "FAIL"
    print(f"  [{icon}] {layer}/{check_name}: {detail}")


print("Funcoes de validacao carregadas.")

## Check 1: Bronze (9 tabelas)

In [ ]:
log("Check 1: Validando camada Bronze...")

bronze_tables = [
    "dados_cadastrais", "telco", "score_bureau_movel",
    "recarga", "pagamento", "faturamento",
    "dim_calendario", "recarga_dim", "faturamento_dim",
]

bronze_counts = {}
for table_name in bronze_tables:
    try:
        uri = f"oci://{BUCKETS['bronze']}@{NAMESPACE}/{table_name}/"
        df = read_table(spark, uri, fmt=STORAGE_FORMAT)
        count = df.count()
        bronze_counts[table_name] = count
        if count > 0:
            add_result(f"{table_name}_exists", "Bronze", "PASS", f"{count:,} rows")
        else:
            add_result(f"{table_name}_exists", "Bronze", "FAIL", "0 rows")
    except Exception as e:
        bronze_counts[table_name] = 0
        add_result(f"{table_name}_exists", "Bronze", "FAIL", str(e))

add_result("table_count", "Bronze",
    "PASS" if len([c for c in bronze_counts.values() if c > 0]) == 9 else "FAIL",
    f"{len([c for c in bronze_counts.values() if c > 0])}/9 tabelas")

print(f"\nBronze: {len([c for c in bronze_counts.values() if c > 0])}/9 tabelas")

## Check 2: Silver rawdata (19 tabelas)

In [ ]:
log("Check 2: Validando camada Silver rawdata...")

silver_tables = list(TABLES_MAIN.keys()) + list(DIMENSION_TABLES.keys())
silver_counts = {}

for table_name in silver_tables:
    pk_cols = TABLES_MAIN.get(table_name, DIMENSION_TABLES.get(table_name, {})).get("pk", [])
    try:
        uri = f"oci://{BUCKETS['silver']}@{NAMESPACE}/rawdata/{table_name}/"
        df = read_table(spark, uri, fmt=STORAGE_FORMAT)
        count = df.count()
        silver_counts[table_name] = count
        if count > 0:
            add_result(f"{table_name}_exists", "Silver", "PASS", f"{count:,} rows")
        else:
            add_result(f"{table_name}_exists", "Silver", "FAIL", "0 rows")
        # PK uniqueness
        if pk_cols:
            valid_pks = [c for c in pk_cols if c in df.columns]
            if valid_pks:
                dup = df.groupBy(*valid_pks).agg(spark_count("*").alias("cnt")).filter(col("cnt") > 1).count()
                add_result(f"{table_name}_pk_unique", "Silver",
                    "PASS" if dup == 0 else "FAIL",
                    f"PK={valid_pks}, dups={dup}")
    except Exception as e:
        silver_counts[table_name] = 0
        add_result(f"{table_name}_exists", "Silver", "FAIL", str(e))

silver_found = len([c for c in silver_counts.values() if c > 0])
add_result("table_count", "Silver",
    "PASS" if silver_found == 19 else "FAIL",
    f"{silver_found}/19 tabelas")

print(f"\nSilver: {silver_found}/19 tabelas")

## Check 3: Gold Books (3 books)

In [ ]:
log("Check 3: Validando Gold books...")

book_configs = {
    "book_recarga_cmv": {"prefix": "REC_"},
    "book_pagamento": {"prefix": "PAG_"},
    "book_faturamento": {"prefix": "FAT_"},
}

book_counts = {}
for book_name, cfg in book_configs.items():
    prefix = cfg["prefix"]
    try:
        uri = f"oci://{GOLD_BUCKET}@{NAMESPACE}/books/{book_name}/"
        df = read_table(spark, uri, fmt=STORAGE_FORMAT)
        count = df.count()
        book_counts[book_name] = count
        col_count = len(df.columns)
        prefix_cols = len([c for c in df.columns if c.startswith(prefix)])

        add_result(f"{book_name}_exists", "Gold/books",
            "PASS" if count > 0 else "FAIL",
            f"{count:,} rows, {col_count} cols")
        add_result(f"{book_name}_prefix", "Gold/books",
            "PASS" if prefix_cols > 0 else "FAIL",
            f"{prefix_cols} cols com prefixo {prefix}")
        # PK uniqueness
        dup = df.groupBy("NUM_CPF", "SAFRA").agg(spark_count("*").alias("cnt")).filter(col("cnt") > 1).count()
        add_result(f"{book_name}_pk", "Gold/books",
            "PASS" if dup == 0 else "FAIL",
            f"dups={dup}")
    except Exception as e:
        book_counts[book_name] = 0
        add_result(f"{book_name}_exists", "Gold/books", "FAIL", str(e))

print(f"\nGold books: {len([c for c in book_counts.values() if c > 0])}/3 books")

## Check 4: Gold Consolidated

In [ ]:
log("Check 4: Validando Gold consolidated...")

try:
    df_gold = read_table(spark, GOLD_CONSOLIDATED_PATH, fmt=STORAGE_FORMAT)
    gold_cols = df_gold.columns
    gold_rows = df_gold.count()
    gold_col_count = len(gold_cols)

    add_result("row_count", "Gold/consolidated", "PASS" if gold_rows > 0 else "FAIL",
        f"{gold_rows:,} rows")
    add_result("col_count", "Gold/consolidated", "PASS",
        f"{gold_col_count} cols")

    # SAFRA completeness
    actual_safras = sorted([r.SAFRA for r in df_gold.select("SAFRA").distinct().collect()])
    add_result("safra_completeness", "Gold/consolidated",
        "PASS" if actual_safras == SAFRAS else "FAIL",
        f"SAFRAs={actual_safras}")

    # Leakage check
    leakage = [c for c in gold_cols if any(bl in c for bl in LEAKAGE_BLACKLIST)]
    add_result("leakage_absent", "Gold/consolidated",
        "PASS" if not leakage else "FAIL",
        f"leakage_cols={leakage}" if leakage else "clean")

    # Required columns
    missing_req = [c for c in REQUIRED_COLUMNS if c not in gold_cols]
    add_result("required_columns", "Gold/consolidated",
        "PASS" if not missing_req else "FAIL",
        f"missing={missing_req}" if missing_req else "all present")

    # Prefixes
    for prefix in EXPECTED_PREFIXES:
        pcols = [c for c in gold_cols if c.startswith(prefix)]
        add_result(f"prefix_{prefix.rstrip('_')}", "Gold/consolidated",
            "PASS" if len(pcols) > 0 else "FAIL",
            f"{len(pcols)} cols")

    # PK uniqueness
    dup = df_gold.groupBy("NUM_CPF", "SAFRA").agg(spark_count("*").alias("cnt")).filter(col("cnt") > 1).count()
    add_result("pk_unique", "Gold/consolidated",
        "PASS" if dup == 0 else "FAIL",
        f"dups={dup}")

except Exception as e:
    add_result("access", "Gold/consolidated", "FAIL", str(e))

print(f"\nGold consolidated validado")

## Check 5: Gold Final

In [ ]:
log("Check 5: Validando Gold final...")

try:
    df_final = read_table(spark, GOLD_FINAL_PATH, fmt=STORAGE_FORMAT)
    final_rows = df_final.count()
    final_col_count = len(df_final.columns)

    # Col count == SELECTED_FEATURES + keys (NUM_CPF, SAFRA, FPD)
    expected_cols = len(SELECTED_FEATURES) + 3
    add_result("col_count", "Gold/final",
        "PASS" if final_col_count == expected_cols else "FAIL",
        f"{final_col_count} cols (esperado: {expected_cols})")

    # Row count == consolidated
    add_result("row_count_match", "Gold/final",
        "PASS" if final_rows == gold_rows else "FAIL",
        f"final={final_rows:,} vs consolidated={gold_rows:,}")

    # PK uniqueness
    dup = df_final.groupBy("NUM_CPF", "SAFRA").agg(spark_count("*").alias("cnt")).filter(col("cnt") > 1).count()
    add_result("pk_unique", "Gold/final",
        "PASS" if dup == 0 else "FAIL",
        f"dups={dup}")

except Exception as e:
    add_result("access", "Gold/final", "FAIL", str(e))

print(f"\nGold final validado")

## Check 6: Cross-Layer Consistency

In [ ]:
log("Check 6: Validando consistencia cross-layer...")

# Bronze >= Silver (dedup removes rows)
for table_name in ["dados_cadastrais", "telco", "score_bureau_movel", "recarga", "pagamento", "faturamento"]:
    b_count = bronze_counts.get(table_name, 0)
    s_count = silver_counts.get(table_name, 0)
    if b_count > 0 and s_count > 0:
        add_result(f"{table_name}_bronze_gte_silver", "Cross-layer",
            "PASS" if b_count >= s_count else "FAIL",
            f"Bronze={b_count:,} >= Silver={s_count:,}")

# Gold rows == Silver base rows (dados_cadastrais)
cadastro_silver = silver_counts.get("dados_cadastrais", 0)
try:
    add_result("gold_rows_eq_silver_base", "Cross-layer",
        "PASS" if gold_rows == cadastro_silver else "FAIL",
        f"Gold={gold_rows:,} vs Silver/dados_cadastrais={cadastro_silver:,}")
except NameError:
    add_result("gold_rows_eq_silver_base", "Cross-layer", "FAIL", "Gold not loaded")

print(f"\nCross-layer validado")

## Relatorio Final

In [ ]:
# Relatorio final
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
total = len(results)

print(f"\n{'='*70}")
print(f"RELATORIO DE VALIDACAO — PIPELINE OCI")
print(f"{'='*70}")
print(f"Gerado em: {now}")
print(f"")
print(f"{'#':>3s}  {'Camada':20s} {'Check':35s} {'Status':>6s}  {'Detalhe'}")
print('-' * 90)
for i, r in enumerate(results, 1):
    icon = 'PASS' if r['status'] == 'PASS' else 'FAIL'
    print(f"{i:3d}  {r['layer']:20s} {r['check']:35s} {icon:>6s}  {r['detail']}")

print(f"\n{'='*70}")
print(f"RESULTADO FINAL: {passed} PASS / {failed} FAIL de {total} checks")
if failed == 0:
    print("Pipeline OCI VALIDADO — todas as camadas consistentes.")
else:
    print(f"ATENCAO: {failed} check(s) falharam. Revisar detalhes acima.")
print(f"{'='*70}")

# Salvar relatorio
os.makedirs(ARTIFACT_DIR, exist_ok=True)
report_path = os.path.join(ARTIFACT_DIR, "validation_report.md")
lines = [
    "# Relatorio de Validacao — Pipeline OCI",
    "",
    f"**Gerado em**: {now}",
    f"**Resultado**: {'TODOS PASS' if failed == 0 else f'{failed} FALHAS'} ({passed}/{total})",
    "",
    "| # | Camada | Check | Status | Detalhe |",
    "|---|--------|-------|--------|---------|"]
for i, r in enumerate(results, 1):
    icon = 'PASS' if r['status'] == 'PASS' else '**FAIL**'
    lines.append(f"| {i} | {r['layer']} | {r['check']} | {icon} | {r['detail']} |")
lines.append("")
lines.append(f"---")
lines.append(f"*06_data_quality.ipynb | {total} checks*")

with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
log(f"Relatorio salvo em {report_path}")